# 03 — Feature Engineering: Silver → Gold Books

Construcao dos books de variaveis (feature engineering) a partir das tabelas Silver.

**3 Books de Variaveis**:
- **Book Recarga** (prefixo `REC_`): ~102 variaveis derivadas de transacoes de recarga
- **Book Pagamento** (prefixo `PAG_`): ~154 variaveis derivadas de transacoes de pagamento
- **Book Faturamento** (prefixo `FAT_`): ~108 variaveis derivadas de transacoes de faturamento

**Escrita**: Gold bucket `books/` em formato Delta, particionado por SAFRA.

**Equivalente Fabric**: `4-construcao-books/`

**Hackathon PoD Academy** — Claro + Oracle

In [ ]:
import sys
from datetime import date
from functools import reduce

sys.path.insert(0, "..")
from config import *

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import col, current_timestamp

print(f"NAMESPACE: {NAMESPACE}")
print(f"Silver bucket: {BUCKETS['silver']}")
print(f"Gold bucket: {BUCKETS['gold']}")
print(f"Landing bucket: {BUCKETS['landing']}")
print(f"SAFRAS: {SAFRAS}")
print(f"Storage format: {STORAGE_FORMAT}")

In [ ]:
# Criar SparkSession com configuracao completa (Delta + Tuning)
builder = SparkSession.builder.appName("feature-engineering-gold")
for key, value in SPARK_CONFIG.items():
    builder = builder.config(key, value)
spark = builder.getOrCreate()

print(f"SparkSession criada: {spark.version}")
print(f"AQE enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"Shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Broadcast threshold: {spark.conf.get('spark.sql.autoBroadcastJoinThreshold')}")

## Carregamento das Tabelas Silver

Carregamento das tabelas fato (recarga, pagamento, faturamento) e dimensoes do Silver rawdata como TempViews do Spark SQL.

**Tabelas fato**: Lidas do Silver bucket em formato Delta.
**Tabelas dimensao**: Lidas do Silver rawdata (processadas no NB 02).

In [ ]:
silver_bucket = BUCKETS["silver"]
gold_bucket = BUCKETS["gold"]

def read_silver(spark, table_name):
    """Le tabela do Silver bucket."""
    uri = f"oci://{silver_bucket}@{NAMESPACE}/rawdata/{table_name}/"
    if SILVER_READ_FORMAT == "delta":
        return spark.read.format("delta").load(uri)
    return spark.read.parquet(uri)


def drop_internal_cols(df):
    """Remove colunas internas de auditoria."""
    internal = ["_execution_id", "_data_inclusao", "_data_alteracao_silver"]
    for c in internal:
        if c in df.columns:
            df = df.drop(c)
    return df


def get_select_cols(df, alias, exclude, prefix=None):
    """Gera lista de colunas com renomeacao opcional por prefixo."""
    exclude_lower = {c.lower() for c in exclude}
    cols = []
    for c in df.columns:
        if c.lower() not in exclude_lower:
            if prefix:
                cols.append(f"{alias}.{c} AS {prefix}_{c}")
            else:
                cols.append(f"{alias}.{c}")
    return cols


def build_book_for_safras(spark, sql_template, safras):
    """Executa SQL template para cada SAFRA e retorna DataFrame unificado."""
    dfs = []
    for safra in safras:
        cutoff = safra_to_cutoff(safra)
        query = sql_template.format(safra=safra, data_cutoff=cutoff)
        df = spark.sql(query)
        dfs.append(df)
        print(f"  SAFRA {safra}: query executada")
    if not dfs:
        return None
    result = dfs[0]
    for df in dfs[1:]:
        result = result.unionByName(df, allowMissingColumns=True)
    return result


# --- Carregar tabelas fato ---
print("[GOLD] Carregando tabelas fato do Silver...")

df_recarga = read_silver(spark, "recarga")
df_recarga.createOrReplaceTempView("recarga")
print(f"  recarga: {len(df_recarga.columns)} colunas")

df_pagamento = read_silver(spark, "pagamento")
df_pagamento.createOrReplaceTempView("pagamento")
print(f"  pagamento: {len(df_pagamento.columns)} colunas")

df_faturamento = read_silver(spark, "faturamento")
df_faturamento.createOrReplaceTempView("fato_faturamento")
print(f"  faturamento: {len(df_faturamento.columns)} colunas")

# --- Carregar tabelas de dimensao do Silver rawdata ---
print("\n[GOLD] Carregando tabelas de dimensao do Silver rawdata...")

dim_views = {
    "dim_canal":             "canal_aquisicao_credito",
    "dim_plano":             "plano_preco",
    "dim_tipo_recarga":      "tipo_recarga",
    "dim_tipo_insercao":     "tipo_insercao",
    "dim_forma_pagamento":   "forma_pagamento",
    "dim_instituicao":       "instituicao",
    "dim_plataforma":        "plataforma",
    "dim_tipo_faturamento":  "tipo_faturamento",
}

for view_name, table_name in dim_views.items():
    df_dim = drop_internal_cols(read_silver(spark, table_name))
    df_dim.createOrReplaceTempView(view_name)
    print(f"  {view_name}: {len(df_dim.columns)} colunas, {df_dim.count()} linhas")

print("\n[OK] Todas as tabelas carregadas como TempViews")

## Book Recarga (102 variaveis, prefixo REC_)

Variaveis derivadas de transacoes de recarga, enriquecidas com dimensoes (canal, plano, tipo recarga, tipo insercao, forma pagamento, instituicao).

**Categorias de variaveis**:
- Contagens (QTD_*): recargas, linhas, plataformas, tipos, instituicoes
- Valores (VLR_*): credito, bonus, real — total, medio, max, stddev
- Flags por plataforma, status, cartao, plano, instituicao
- Taxas e ratios derivados
- Score de risco e segmentacao
- Temporais: dias entre recargas, desde ultima/primeira recarga

In [ ]:
SQL_RECARGA = """
WITH
base_enrich AS (
    SELECT
        r.*,
        CAST('{safra}' AS INT) AS SAFRA,
        c.DSC_CANAL_AQUISICAO_BI,
        c.COD_TIPO_CREDITO AS CANAL_TIPO_CREDITO,
        p.DSC_TIPO_PLANO_BI,
        p.DSC_GRUPO_PLANO_BI,
        p.IND_AMDOCS_PLAT_PRE,
        tr.DSC_TIPO_RECARGA,
        ti.DSC_TIPO_INSERCAO,
        fp.DSC_FORMA_PAGAMENTO,
        i.DSC_INSTITUICAO,
        i.DSC_TIPO_INSTITUICAO,
        CASE WHEN r.COD_PLATAFORMA_ATU = 'PREPG' THEN 1 ELSE 0 END AS FLAG_PLAT_PREPG,
        CASE WHEN r.COD_PLATAFORMA_ATU = 'AUTOC' THEN 1 ELSE 0 END AS FLAG_PLAT_AUTOC,
        CASE WHEN r.COD_PLATAFORMA_ATU = 'FLEXD' THEN 1 ELSE 0 END AS FLAG_PLAT_FLEXD,
        CASE WHEN r.COD_PLATAFORMA_ATU = 'CTLFC' THEN 1 ELSE 0 END AS FLAG_PLAT_CTLFC,
        CASE WHEN r.COD_PLATAFORMA_ATU = 'POSPG' THEN 1 ELSE 0 END AS FLAG_PLAT_POSPG,
        CASE WHEN r.COD_STATUS_PLATAFORMA = 'A' THEN 1 ELSE 0 END AS FLAG_STATUS_A,
        CASE WHEN r.COD_STATUS_PLATAFORMA = 'ZB1' THEN 1 ELSE 0 END AS FLAG_STATUS_ZB1,
        CASE WHEN r.COD_STATUS_PLATAFORMA = 'ZB2' THEN 1 ELSE 0 END AS FLAG_STATUS_ZB2,
        CASE WHEN r.COD_STATUS_PLATAFORMA = 'NDF' THEN 1 ELSE 0 END AS FLAG_STATUS_NDF,
        CASE WHEN r.DSC_GRUPO_CARTAO_WPP = 'NaoSeAplica' THEN 1 ELSE 0 END AS FLAG_CARTAO_NA,
        CASE WHEN r.DSC_GRUPO_CARTAO_WPP = 'Rec.Online' THEN 1 ELSE 0 END AS FLAG_CARTAO_ONLINE,
        CASE WHEN r.DSC_GRUPO_CARTAO_WPP = 'AtivPromocao' THEN 1 ELSE 0 END AS FLAG_CARTAO_PROMO,
        CASE WHEN r.DSC_GRUPO_CARTAO_WPP LIKE 'ChipPre%' THEN 1 ELSE 0 END AS FLAG_CARTAO_CHIPPRE,
        CASE WHEN p.DSC_TIPO_PLANO_BI = 'Varejo' THEN 1 ELSE 0 END AS FLAG_PLANO_VAREJO,
        CASE WHEN p.DSC_TIPO_PLANO_BI = 'Corporativos' THEN 1 ELSE 0 END AS FLAG_PLANO_CORP,
        CASE WHEN p.DSC_TIPO_PLANO_BI = 'Mid' THEN 1 ELSE 0 END AS FLAG_PLANO_MID,
        CASE WHEN i.DSC_TIPO_INSTITUICAO = 'Distribuidor Regional' THEN 1 ELSE 0 END AS FLAG_INST_DIST_REG,
        CASE WHEN i.DSC_TIPO_INSTITUICAO = 'Street Seller' THEN 1 ELSE 0 END AS FLAG_INST_STREET,
        CASE WHEN i.DSC_TIPO_INSTITUICAO = 'Venda Direta' THEN 1 ELSE 0 END AS FLAG_INST_DIRETA,
        CASE WHEN i.DSC_TIPO_INSTITUICAO = 'Varejo' THEN 1 ELSE 0 END AS FLAG_INST_VAREJO,
        COALESCE(r.FLAG_SOS, 0) AS FLAG_SOS_CLEAN,
        CASE WHEN r.COD_PLATAFORMA_ATU IN ('CTLFC', 'FLEXD') THEN 1 ELSE 0 END AS FLAG_PLAT_CONTROLE
    FROM recarga r
    LEFT JOIN dim_canal c ON r.COD_CANAL_AQUISICAO = c.COD_CANAL_AQUISICAO
    LEFT JOIN dim_plano p ON r.DW_PLANO_TARIFACAO = p.DW_PLANO
    LEFT JOIN dim_tipo_recarga tr ON r.DW_TIPO_RECARGA = tr.DW_TIPO_RECARGA
    LEFT JOIN dim_tipo_insercao ti ON r.DW_TIPO_INSERCAO = ti.DW_TIPO_INSERCAO
    LEFT JOIN dim_forma_pagamento fp ON r.DW_FORMA_PAGAMENTO = fp.DW_FORMA_PAGAMENTO
    LEFT JOIN dim_instituicao i ON r.DW_INSTITUICAO = i.DW_INSTITUICAO
    WHERE r.DAT_INSERCAO_CREDITO < '{data_cutoff}'
),
agregado AS (
    SELECT
        SAFRA, NUM_CPF,
        COUNT(*) AS QTD_RECARGAS_TOTAL,
        COUNT(DISTINCT DW_NUM_NTC) AS QTD_LINHAS,
        COUNT(DISTINCT DW_NUM_CLIENTE) AS QTD_CLIENTES_DW,
        COUNT(DISTINCT COD_PLATAFORMA_ATU) AS QTD_PLATAFORMAS,
        COUNT(DISTINCT DW_TIPO_RECARGA) AS QTD_TIPOS_RECARGA,
        COUNT(DISTINCT DW_TIPO_INSERCAO) AS QTD_TIPOS_INSERCAO,
        COUNT(DISTINCT DW_FORMA_PAGAMENTO) AS QTD_FORMAS_PAGTO,
        COUNT(DISTINCT DW_INSTITUICAO) AS QTD_INSTITUICOES,
        COUNT(DISTINCT DW_PLANO_TARIFACAO) AS QTD_PLANOS,
        COUNT(DISTINCT DATE(DAT_INSERCAO_CREDITO)) AS QTD_DIAS_RECARGA,
        SUM(COALESCE(VAL_CREDITO_INSERIDO, 0)) AS VLR_CREDITO_TOTAL,
        AVG(COALESCE(VAL_CREDITO_INSERIDO, 0)) AS VLR_CREDITO_MEDIO,
        MAX(COALESCE(VAL_CREDITO_INSERIDO, 0)) AS VLR_CREDITO_MAX,
        MIN(CASE WHEN VAL_CREDITO_INSERIDO > 0 THEN VAL_CREDITO_INSERIDO END) AS VLR_CREDITO_MIN,
        STDDEV(COALESCE(VAL_CREDITO_INSERIDO, 0)) AS VLR_CREDITO_STDDEV,
        SUM(COALESCE(VAL_BONUS, 0)) AS VLR_BONUS_TOTAL,
        AVG(COALESCE(VAL_BONUS, 0)) AS VLR_BONUS_MEDIO,
        MAX(COALESCE(VAL_BONUS, 0)) AS VLR_BONUS_MAX,
        SUM(COALESCE(VAL_REAL, 0)) AS VLR_REAL_TOTAL,
        AVG(COALESCE(VAL_REAL, 0)) AS VLR_REAL_MEDIO,
        MAX(COALESCE(VAL_REAL, 0)) AS VLR_REAL_MAX,
        STDDEV(COALESCE(VAL_REAL, 0)) AS VLR_REAL_STDDEV,
        SUM(FLAG_PLAT_PREPG) AS QTD_PLAT_PREPG,
        SUM(FLAG_PLAT_AUTOC) AS QTD_PLAT_AUTOC,
        SUM(FLAG_PLAT_FLEXD) AS QTD_PLAT_FLEXD,
        SUM(FLAG_PLAT_CTLFC) AS QTD_PLAT_CTLFC,
        SUM(FLAG_PLAT_POSPG) AS QTD_PLAT_POSPG,
        SUM(CASE WHEN FLAG_PLAT_PREPG = 1 THEN COALESCE(VAL_CREDITO_INSERIDO, 0) ELSE 0 END) AS VLR_PLAT_PREPG,
        SUM(CASE WHEN FLAG_PLAT_AUTOC = 1 THEN COALESCE(VAL_CREDITO_INSERIDO, 0) ELSE 0 END) AS VLR_PLAT_AUTOC,
        SUM(CASE WHEN FLAG_PLAT_FLEXD = 1 THEN COALESCE(VAL_CREDITO_INSERIDO, 0) ELSE 0 END) AS VLR_PLAT_FLEXD,
        SUM(CASE WHEN FLAG_PLAT_CTLFC = 1 THEN COALESCE(VAL_CREDITO_INSERIDO, 0) ELSE 0 END) AS VLR_PLAT_CTLFC,
        SUM(FLAG_PLAT_CONTROLE) AS QTD_PLAT_CONTROLE,
        SUM(FLAG_STATUS_A) AS QTD_STATUS_A,
        SUM(FLAG_STATUS_ZB1) AS QTD_STATUS_ZB1,
        SUM(FLAG_STATUS_ZB2) AS QTD_STATUS_ZB2,
        SUM(FLAG_STATUS_NDF) AS QTD_STATUS_NDF,
        SUM(CASE WHEN FLAG_STATUS_A = 1 THEN COALESCE(VAL_CREDITO_INSERIDO, 0) ELSE 0 END) AS VLR_STATUS_A,
        SUM(CASE WHEN FLAG_STATUS_ZB1 = 1 THEN COALESCE(VAL_CREDITO_INSERIDO, 0) ELSE 0 END) AS VLR_STATUS_ZB1,
        SUM(CASE WHEN FLAG_STATUS_ZB2 = 1 THEN COALESCE(VAL_CREDITO_INSERIDO, 0) ELSE 0 END) AS VLR_STATUS_ZB2,
        SUM(FLAG_CARTAO_NA) AS QTD_CARTAO_NA,
        SUM(FLAG_CARTAO_ONLINE) AS QTD_CARTAO_ONLINE,
        SUM(FLAG_CARTAO_PROMO) AS QTD_CARTAO_PROMO,
        SUM(FLAG_CARTAO_CHIPPRE) AS QTD_CARTAO_CHIPPRE,
        SUM(CASE WHEN FLAG_CARTAO_ONLINE = 1 THEN COALESCE(VAL_CREDITO_INSERIDO, 0) ELSE 0 END) AS VLR_CARTAO_ONLINE,
        SUM(CASE WHEN FLAG_CARTAO_PROMO = 1 THEN COALESCE(VAL_CREDITO_INSERIDO, 0) ELSE 0 END) AS VLR_CARTAO_PROMO,
        SUM(FLAG_PLANO_VAREJO) AS QTD_PLANO_VAREJO,
        SUM(FLAG_PLANO_CORP) AS QTD_PLANO_CORP,
        SUM(FLAG_PLANO_MID) AS QTD_PLANO_MID,
        SUM(CASE WHEN FLAG_PLANO_VAREJO = 1 THEN COALESCE(VAL_CREDITO_INSERIDO, 0) ELSE 0 END) AS VLR_PLANO_VAREJO,
        SUM(CASE WHEN FLAG_PLANO_CORP = 1 THEN COALESCE(VAL_CREDITO_INSERIDO, 0) ELSE 0 END) AS VLR_PLANO_CORP,
        SUM(FLAG_INST_DIST_REG) AS QTD_INST_DIST_REG,
        SUM(FLAG_INST_STREET) AS QTD_INST_STREET,
        SUM(FLAG_INST_DIRETA) AS QTD_INST_DIRETA,
        SUM(FLAG_INST_VAREJO) AS QTD_INST_VAREJO,
        SUM(CASE WHEN FLAG_INST_STREET = 1 THEN COALESCE(VAL_CREDITO_INSERIDO, 0) ELSE 0 END) AS VLR_INST_STREET,
        SUM(CASE WHEN FLAG_INST_DIRETA = 1 THEN COALESCE(VAL_CREDITO_INSERIDO, 0) ELSE 0 END) AS VLR_INST_DIRETA,
        SUM(FLAG_SOS_CLEAN) AS QTD_SOS,
        SUM(CASE WHEN FLAG_SOS_CLEAN = 1 THEN COALESCE(VAL_CREDITO_INSERIDO, 0) ELSE 0 END) AS VLR_SOS_TOTAL,
        MIN(DAT_INSERCAO_CREDITO) AS DT_PRIMEIRA_RECARGA,
        MAX(DAT_INSERCAO_CREDITO) AS DT_ULTIMA_RECARGA,
        DATEDIFF(MAX(DAT_INSERCAO_CREDITO), MIN(DAT_INSERCAO_CREDITO)) AS DIAS_ENTRE_RECARGAS,
        DATEDIFF(TO_DATE('{data_cutoff}'), MAX(DAT_INSERCAO_CREDITO)) AS DIAS_DESDE_ULTIMA_RECARGA,
        DATEDIFF(TO_DATE('{data_cutoff}'), MIN(DAT_INSERCAO_CREDITO)) AS DIAS_DESDE_PRIMEIRA_RECARGA,
        COUNT(DISTINCT DATE_FORMAT(DAT_INSERCAO_CREDITO, 'yyyyMM')) AS QTD_MESES_ATIVOS,
        FIRST(DSC_TIPO_PLANO_BI) AS TIPO_PLANO_PRINCIPAL,
        FIRST(DSC_TIPO_INSTITUICAO) AS TIPO_INSTITUICAO_PRINCIPAL,
        FIRST(DSC_GRUPO_CARTAO_WPP) AS GRUPO_CARTAO_PRINCIPAL,
        FIRST(COD_PLATAFORMA_ATU) AS PLATAFORMA_PRINCIPAL
    FROM base_enrich
    GROUP BY SAFRA, NUM_CPF
)
SELECT
    a.*,
    a.QTD_PLAT_PREPG / NULLIF(CAST(a.QTD_RECARGAS_TOTAL AS DOUBLE), 0) AS TAXA_PLAT_PREPG,
    a.QTD_PLAT_AUTOC / NULLIF(CAST(a.QTD_RECARGAS_TOTAL AS DOUBLE), 0) AS TAXA_PLAT_AUTOC,
    a.QTD_PLAT_CONTROLE / NULLIF(CAST(a.QTD_RECARGAS_TOTAL AS DOUBLE), 0) AS TAXA_PLAT_CONTROLE,
    a.QTD_STATUS_A / NULLIF(CAST(a.QTD_RECARGAS_TOTAL AS DOUBLE), 0) AS TAXA_STATUS_A,
    a.QTD_STATUS_ZB1 / NULLIF(CAST(a.QTD_RECARGAS_TOTAL AS DOUBLE), 0) AS TAXA_STATUS_ZB1,
    a.QTD_STATUS_ZB2 / NULLIF(CAST(a.QTD_RECARGAS_TOTAL AS DOUBLE), 0) AS TAXA_STATUS_ZB2,
    a.QTD_CARTAO_ONLINE / NULLIF(CAST(a.QTD_RECARGAS_TOTAL AS DOUBLE), 0) AS TAXA_CARTAO_ONLINE,
    a.QTD_CARTAO_PROMO / NULLIF(CAST(a.QTD_RECARGAS_TOTAL AS DOUBLE), 0) AS TAXA_CARTAO_PROMO,
    a.QTD_SOS / NULLIF(CAST(a.QTD_RECARGAS_TOTAL AS DOUBLE), 0) AS TAXA_SOS,
    a.VLR_PLAT_PREPG / NULLIF(a.VLR_CREDITO_TOTAL, 0) AS SHARE_PLAT_PREPG,
    a.VLR_PLAT_AUTOC / NULLIF(a.VLR_CREDITO_TOTAL, 0) AS SHARE_PLAT_AUTOC,
    a.VLR_CARTAO_ONLINE / NULLIF(a.VLR_CREDITO_TOTAL, 0) AS SHARE_CARTAO_ONLINE,
    a.VLR_CREDITO_STDDEV / NULLIF(a.VLR_CREDITO_MEDIO, 0) AS COEF_VARIACAO_CREDITO,
    a.VLR_REAL_STDDEV / NULLIF(a.VLR_REAL_MEDIO, 0) AS COEF_VARIACAO_REAL,
    a.VLR_CREDITO_MAX / NULLIF(a.VLR_CREDITO_TOTAL, 0) AS INDICE_CONCENTRACAO_CREDITO,
    a.VLR_CREDITO_TOTAL / NULLIF(a.QTD_LINHAS, 0) AS VLR_TICKET_MEDIO_LINHA,
    a.QTD_RECARGAS_TOTAL / NULLIF(GREATEST(a.DIAS_ENTRE_RECARGAS, 1), 0) AS FREQ_RECARGA_DIARIA,
    a.VLR_BONUS_TOTAL / NULLIF(a.VLR_CREDITO_TOTAL, 0) AS RATIO_BONUS_CREDITO,
    LEAST(100, GREATEST(0,
        COALESCE(a.QTD_PLAT_CONTROLE / NULLIF(CAST(a.QTD_RECARGAS_TOTAL AS DOUBLE), 0), 0) * 30 +
        COALESCE((a.QTD_STATUS_ZB1 + a.QTD_STATUS_ZB2) / NULLIF(CAST(a.QTD_RECARGAS_TOTAL AS DOUBLE), 0), 0) * 25 +
        LEAST(1, COALESCE(a.QTD_SOS / NULLIF(CAST(a.QTD_RECARGAS_TOTAL AS DOUBLE), 0), 0) * 3) * 20 +
        LEAST(1, COALESCE(a.VLR_CREDITO_STDDEV / NULLIF(a.VLR_CREDITO_MEDIO, 0), 0)) * 15 +
        LEAST(1, COALESCE(a.VLR_CREDITO_MAX / NULLIF(a.VLR_CREDITO_TOTAL, 0), 0)) * 10
    )) AS SCORE_RISCO,
    CASE
        WHEN a.QTD_PLAT_CONTROLE > a.QTD_RECARGAS_TOTAL * 0.5 THEN 1
        WHEN (a.QTD_STATUS_ZB1 + a.QTD_STATUS_ZB2) > a.QTD_RECARGAS_TOTAL * 0.3 THEN 1
        WHEN a.QTD_SOS > a.QTD_RECARGAS_TOTAL * 0.2 THEN 1
        ELSE 0
    END AS FLAG_ALTO_RISCO,
    CASE
        WHEN a.QTD_PLAT_PREPG = a.QTD_RECARGAS_TOTAL
            AND a.QTD_STATUS_A > a.QTD_RECARGAS_TOTAL * 0.95
            AND a.QTD_SOS = 0 THEN 1
        ELSE 0
    END AS FLAG_BAIXO_RISCO,
    CASE
        WHEN a.QTD_PLAT_CONTROLE > a.QTD_RECARGAS_TOTAL * 0.5
            OR (a.QTD_STATUS_ZB1 + a.QTD_STATUS_ZB2) > a.QTD_RECARGAS_TOTAL * 0.3 THEN 'CRITICO'
        WHEN a.QTD_PLAT_CONTROLE > 0
            OR a.QTD_SOS > a.QTD_RECARGAS_TOTAL * 0.1 THEN 'ALTO'
        WHEN a.QTD_STATUS_ZB1 > 0 OR a.QTD_SOS > 0 THEN 'MEDIO'
        ELSE 'BAIXO'
    END AS SEGMENTO_RISCO
FROM agregado a
"""

In [ ]:
# Executar Book Recarga para todas as SAFRAs
print("[GOLD] Construindo book_recarga_cmv...")
df_book_rec = build_book_for_safras(spark, SQL_RECARGA, SAFRAS)

book_rec_uri = f"oci://{gold_bucket}@{NAMESPACE}/books/book_recarga_cmv/"
writer = df_book_rec.write.mode("overwrite").partitionBy("SAFRA")
if STORAGE_FORMAT == "delta":
    writer.format("delta").option("overwriteSchema", "true").save(book_rec_uri)
else:
    writer.parquet(book_rec_uri)

print(f"[GOLD BOOK] book_recarga_cmv: escrito, {len(df_book_rec.columns)} colunas")

# Re-ler do Delta (evita recomputar DataFrame lazy)
df_book_rec = spark.read.format("delta").load(book_rec_uri)
print(f"[GOLD] Re-leitura book_recarga_cmv: {len(df_book_rec.columns)} colunas")

## Book Pagamento (154 variaveis, prefixo PAG_)

Variaveis derivadas de transacoes de pagamento, enriquecidas com dimensoes (forma pagamento, instituicao/banco, tipo faturamento).

**Categorias de variaveis**:
- Contagens: pagamentos, contratos, faturas, formas pagamento, bancos
- Valores: pagamento fatura, item, credito, atual, original, baixa, juros/multas
- Flags por status pagamento (R/C/P/B), status fatura, forma pagamento, tipo pagamento, alocacao credito
- Taxas e ratios: inadimplencia, juros sobre pagamento, concentracao
- Score de risco e segmentacao
- Temporais: dias entre faturas, desde ultima fatura, medio baixa/deposito

In [ ]:
SQL_PAGAMENTO = """
WITH
base_enrich AS (
    SELECT
        p.*,
        CAST('{safra}' AS INT) AS SAFRA,
        fp.DSC_FORMA_PAGAMENTO,
        i.DSC_INSTITUICAO, i.DSC_TIPO_INSTITUICAO,
        tf.DSC_TIPO_FATURAMENTO, tf.DSC_TIPO_FATURAMENTO_ABREV,
        CASE WHEN p.IND_STATUS_PAGAMENTO = 'R' THEN 1 ELSE 0 END AS FLAG_STATUS_PAGAMENTO_R,
        CASE WHEN p.IND_STATUS_PAGAMENTO = 'C' THEN 1 ELSE 0 END AS FLAG_STATUS_PAGAMENTO_C,
        CASE WHEN p.IND_STATUS_PAGAMENTO = 'P' THEN 1 ELSE 0 END AS FLAG_STATUS_PAGAMENTO_P,
        CASE WHEN p.IND_STATUS_PAGAMENTO = 'B' THEN 1 ELSE 0 END AS FLAG_STATUS_PAGAMENTO_B,
        CASE WHEN p.IND_STATUS_FATURA = 'C' THEN 1 ELSE 0 END AS FLAG_FATURA_FECHADA,
        CASE WHEN p.IND_STATUS_FATURA = 'O' THEN 1 ELSE 0 END AS FLAG_FATURA_ABERTA,
        CASE WHEN p.COD_FORMA_PAGAMENTO = 'CA' THEN 1 ELSE 0 END AS FLAG_FORMA_CA,
        CASE WHEN p.COD_FORMA_PAGAMENTO = 'PB' THEN 1 ELSE 0 END AS FLAG_FORMA_PB,
        CASE WHEN p.COD_FORMA_PAGAMENTO = 'DD' THEN 1 ELSE 0 END AS FLAG_FORMA_DD,
        CASE WHEN p.COD_FORMA_PAGAMENTO = 'PA' THEN 1 ELSE 0 END AS FLAG_FORMA_PA,
        CASE WHEN p.COD_TIPO_PAGAMENTO = 'O' THEN 1 ELSE 0 END AS FLAG_TIPO_O,
        CASE WHEN p.COD_TIPO_PAGAMENTO = 'P' THEN 1 ELSE 0 END AS FLAG_TIPO_P,
        CASE WHEN p.COD_TIPO_PAGAMENTO = 'D' THEN 1 ELSE 0 END AS FLAG_TIPO_D,
        CASE WHEN p.COD_TIPO_PAGAMENTO = 'B' THEN 1 ELSE 0 END AS FLAG_TIPO_B,
        CASE WHEN p.COD_ALOCACAO_CREDITO = 'PYM' THEN 1 ELSE 0 END AS FLAG_ALOCACAO_PYM,
        CASE WHEN p.COD_ALOCACAO_CREDITO = 'CRT' THEN 1 ELSE 0 END AS FLAG_ALOCACAO_CRT,
        CASE WHEN p.COD_ALOCACAO_CREDITO = 'CRTW' THEN 1 ELSE 0 END AS FLAG_ALOCACAO_CRTW
    FROM pagamento p
    LEFT JOIN dim_forma_pagamento fp ON p.DW_FORMA_PAGAMENTO = fp.DW_FORMA_PAGAMENTO
    LEFT JOIN dim_instituicao i ON p.DW_BANCO = i.DW_INSTITUICAO
    LEFT JOIN dim_tipo_faturamento tf ON p.DW_TIPO_FATURA = tf.DW_TIPO_FATURAMENTO
    WHERE p.DAT_STATUS_FATURA < '{data_cutoff}'
),
agregado AS (
    SELECT
        SAFRA, NUM_CPF,
        COUNT(*) AS QTD_PAGAMENTOS_TOTAL,
        COUNT(DISTINCT CONTRATO) AS QTD_CONTRATOS,
        COUNT(DISTINCT SEQ_FATURA) AS QTD_FATURAS_DISTINTAS,
        COUNT(DISTINCT DW_FORMA_PAGAMENTO) AS QTD_FORMAS_PAGAMENTO,
        COUNT(DISTINCT DW_BANCO) AS QTD_BANCOS,
        COUNT(DISTINCT DW_TIPO_FATURA) AS QTD_TIPOS_FATURA,
        COUNT(DISTINCT DW_UN_NEGOCIO) AS QTD_UN_NEGOCIOS,
        COUNT(DISTINCT DW_AREA) AS QTD_AREAS,
        SUM(COALESCE(VAL_PAGAMENTO_FATURA, 0)) AS VLR_PAGAMENTO_FATURA_TOTAL,
        AVG(COALESCE(VAL_PAGAMENTO_FATURA, 0)) AS VLR_PAGAMENTO_FATURA_MEDIO,
        MAX(COALESCE(VAL_PAGAMENTO_FATURA, 0)) AS VLR_PAGAMENTO_FATURA_MAX,
        STDDEV(COALESCE(VAL_PAGAMENTO_FATURA, 0)) AS VLR_PAGAMENTO_FATURA_STDDEV,
        SUM(COALESCE(VAL_PAGAMENTO_ITEM, 0)) AS VLR_PAGAMENTO_ITEM_TOTAL,
        AVG(COALESCE(VAL_PAGAMENTO_ITEM, 0)) AS VLR_PAGAMENTO_ITEM_MEDIO,
        SUM(COALESCE(VAL_PAGAMENTO_CREDITO, 0)) AS VLR_PAGAMENTO_CREDITO_TOTAL,
        AVG(COALESCE(VAL_PAGAMENTO_CREDITO, 0)) AS VLR_PAGAMENTO_CREDITO_MEDIO,
        MAX(COALESCE(VAL_PAGAMENTO_CREDITO, 0)) AS VLR_PAGAMENTO_CREDITO_MAX,
        SUM(COALESCE(VAL_ATUAL_PAGAMENTO, 0)) AS VLR_ATUAL_PAGAMENTO_TOTAL,
        AVG(COALESCE(VAL_ATUAL_PAGAMENTO, 0)) AS VLR_ATUAL_PAGAMENTO_MEDIO,
        SUM(COALESCE(VAL_ORIGINAL_PAGAMENTO, 0)) AS VLR_ORIGINAL_PAGAMENTO_TOTAL,
        AVG(COALESCE(VAL_ORIGINAL_PAGAMENTO, 0)) AS VLR_ORIGINAL_PAGAMENTO_MEDIO,
        SUM(COALESCE(VAL_BAIXA_ATIVIDADE, 0)) AS VLR_BAIXA_ATIVIDADE_TOTAL,
        AVG(COALESCE(VAL_BAIXA_ATIVIDADE, 0)) AS VLR_BAIXA_ATIVIDADE_MEDIO,
        SUM(COALESCE(VAL_JUROS_MULTAS_ITEM, 0)) AS VLR_JUROS_MULTAS_TOTAL,
        AVG(COALESCE(VAL_JUROS_MULTAS_ITEM, 0)) AS VLR_JUROS_MULTAS_MEDIO,
        MAX(COALESCE(VAL_JUROS_MULTAS_ITEM, 0)) AS VLR_JUROS_MULTAS_MAX,
        COUNT(CASE WHEN VAL_JUROS_MULTAS_ITEM > 0 THEN 1 END) AS QTD_PAGAMENTOS_COM_JUROS,
        SUM(COALESCE(VAL_MULTA_EQUIP_ITEM, 0)) AS VLR_MULTA_EQUIP_TOTAL,
        COUNT(CASE WHEN VAL_MULTA_EQUIP_ITEM > 0 THEN 1 END) AS QTD_MULTAS_EQUIP,
        SUM(FLAG_STATUS_PAGAMENTO_R) AS QTD_STATUS_R,
        SUM(FLAG_STATUS_PAGAMENTO_C) AS QTD_STATUS_C,
        SUM(FLAG_STATUS_PAGAMENTO_P) AS QTD_STATUS_P,
        SUM(FLAG_STATUS_PAGAMENTO_B) AS QTD_STATUS_B,
        SUM(CASE WHEN FLAG_STATUS_PAGAMENTO_R = 1 THEN COALESCE(VAL_PAGAMENTO_FATURA, 0) ELSE 0 END) AS VLR_STATUS_R,
        SUM(CASE WHEN FLAG_STATUS_PAGAMENTO_C = 1 THEN COALESCE(VAL_PAGAMENTO_FATURA, 0) ELSE 0 END) AS VLR_STATUS_C,
        SUM(CASE WHEN FLAG_STATUS_PAGAMENTO_P = 1 THEN COALESCE(VAL_PAGAMENTO_FATURA, 0) ELSE 0 END) AS VLR_STATUS_P,
        SUM(CASE WHEN FLAG_STATUS_PAGAMENTO_B = 1 THEN COALESCE(VAL_PAGAMENTO_FATURA, 0) ELSE 0 END) AS VLR_STATUS_B,
        SUM(FLAG_FATURA_FECHADA) AS QTD_FATURA_FECHADA,
        SUM(FLAG_FATURA_ABERTA) AS QTD_FATURA_ABERTA,
        SUM(CASE WHEN FLAG_FATURA_FECHADA = 1 THEN COALESCE(VAL_PAGAMENTO_FATURA, 0) ELSE 0 END) AS VLR_FATURA_FECHADA,
        SUM(CASE WHEN FLAG_FATURA_ABERTA = 1 THEN COALESCE(VAL_PAGAMENTO_FATURA, 0) ELSE 0 END) AS VLR_FATURA_ABERTA,
        SUM(FLAG_FORMA_CA) AS QTD_FORMA_CA,
        SUM(FLAG_FORMA_PB) AS QTD_FORMA_PB,
        SUM(FLAG_FORMA_DD) AS QTD_FORMA_DD,
        SUM(FLAG_FORMA_PA) AS QTD_FORMA_PA,
        SUM(CASE WHEN FLAG_FORMA_CA = 1 THEN COALESCE(VAL_PAGAMENTO_FATURA, 0) ELSE 0 END) AS VLR_FORMA_CA,
        SUM(CASE WHEN FLAG_FORMA_PB = 1 THEN COALESCE(VAL_PAGAMENTO_FATURA, 0) ELSE 0 END) AS VLR_FORMA_PB,
        SUM(CASE WHEN FLAG_FORMA_DD = 1 THEN COALESCE(VAL_PAGAMENTO_FATURA, 0) ELSE 0 END) AS VLR_FORMA_DD,
        SUM(CASE WHEN FLAG_FORMA_PA = 1 THEN COALESCE(VAL_PAGAMENTO_FATURA, 0) ELSE 0 END) AS VLR_FORMA_PA,
        SUM(FLAG_TIPO_O) AS QTD_TIPO_O,
        SUM(FLAG_TIPO_P) AS QTD_TIPO_P,
        SUM(FLAG_TIPO_D) AS QTD_TIPO_D,
        SUM(FLAG_TIPO_B) AS QTD_TIPO_B,
        SUM(CASE WHEN FLAG_TIPO_O = 1 THEN COALESCE(VAL_PAGAMENTO_FATURA, 0) ELSE 0 END) AS VLR_TIPO_O,
        SUM(CASE WHEN FLAG_TIPO_P = 1 THEN COALESCE(VAL_PAGAMENTO_FATURA, 0) ELSE 0 END) AS VLR_TIPO_P,
        SUM(CASE WHEN FLAG_TIPO_D = 1 THEN COALESCE(VAL_PAGAMENTO_FATURA, 0) ELSE 0 END) AS VLR_TIPO_D,
        SUM(CASE WHEN FLAG_TIPO_B = 1 THEN COALESCE(VAL_PAGAMENTO_FATURA, 0) ELSE 0 END) AS VLR_TIPO_B,
        SUM(FLAG_ALOCACAO_PYM) AS QTD_ALOCACAO_PYM,
        SUM(FLAG_ALOCACAO_CRT) AS QTD_ALOCACAO_CRT,
        SUM(FLAG_ALOCACAO_CRTW) AS QTD_ALOCACAO_CRTW,
        SUM(CASE WHEN FLAG_ALOCACAO_PYM = 1 THEN COALESCE(VAL_PAGAMENTO_CREDITO, 0) ELSE 0 END) AS VLR_ALOCACAO_PYM,
        SUM(CASE WHEN FLAG_ALOCACAO_CRT = 1 THEN COALESCE(VAL_PAGAMENTO_CREDITO, 0) ELSE 0 END) AS VLR_ALOCACAO_CRT,
        MIN(DAT_STATUS_FATURA) AS DT_PRIMEIRA_FATURA,
        MAX(DAT_STATUS_FATURA) AS DT_ULTIMA_FATURA,
        MIN(DAT_CRIACAO_DW) AS DT_PRIMEIRA_CRIACAO_DW,
        MAX(DAT_CRIACAO_DW) AS DT_ULTIMA_CRIACAO_DW,
        DATEDIFF(MAX(DAT_STATUS_FATURA), MIN(DAT_STATUS_FATURA)) AS DIAS_ENTRE_FATURAS,
        DATEDIFF(TO_DATE('{data_cutoff}'), MAX(DAT_STATUS_FATURA)) AS DIAS_DESDE_ULTIMA_FATURA,
        AVG(DATEDIFF(DAT_BAIXA_ATIVIDADE, DAT_CRIACAO_ATIVIDADE)) AS DIAS_MEDIO_BAIXA,
        AVG(DATEDIFF(DAT_DEPOSITO_ATIVIDADE, DAT_CRIACAO_ATIVIDADE)) AS DIAS_MEDIO_DEPOSITO,
        COUNT(DISTINCT DATE_FORMAT(DAT_STATUS_FATURA, 'yyyyMM')) AS QTD_MESES_ATIVOS,
        FIRST(DSC_FORMA_PAGAMENTO) AS FORMA_PAGAMENTO_PRINCIPAL,
        FIRST(DSC_TIPO_INSTITUICAO) AS TIPO_INSTITUICAO_PRINCIPAL,
        FIRST(DSC_TIPO_FATURAMENTO_ABREV) AS TIPO_FATURAMENTO_PRINCIPAL,
        COUNT(DISTINCT DSC_TIPO_INSTITUICAO) AS QTD_TIPOS_INSTITUICAO
    FROM base_enrich
    GROUP BY SAFRA, NUM_CPF
)
SELECT
    a.*,
    a.QTD_STATUS_R / NULLIF(CAST(a.QTD_PAGAMENTOS_TOTAL AS DOUBLE), 0) AS TAXA_STATUS_R,
    a.QTD_STATUS_C / NULLIF(CAST(a.QTD_PAGAMENTOS_TOTAL AS DOUBLE), 0) AS TAXA_STATUS_C,
    a.QTD_STATUS_P / NULLIF(CAST(a.QTD_PAGAMENTOS_TOTAL AS DOUBLE), 0) AS TAXA_STATUS_P,
    a.QTD_STATUS_B / NULLIF(CAST(a.QTD_PAGAMENTOS_TOTAL AS DOUBLE), 0) AS TAXA_STATUS_B,
    a.QTD_FORMA_CA / NULLIF(CAST(a.QTD_PAGAMENTOS_TOTAL AS DOUBLE), 0) AS TAXA_FORMA_CA,
    a.QTD_FORMA_PB / NULLIF(CAST(a.QTD_PAGAMENTOS_TOTAL AS DOUBLE), 0) AS TAXA_FORMA_PB,
    a.QTD_FORMA_DD / NULLIF(CAST(a.QTD_PAGAMENTOS_TOTAL AS DOUBLE), 0) AS TAXA_FORMA_DD,
    a.QTD_FORMA_PA / NULLIF(CAST(a.QTD_PAGAMENTOS_TOTAL AS DOUBLE), 0) AS TAXA_FORMA_PA,
    a.QTD_FATURA_ABERTA / NULLIF(CAST(a.QTD_PAGAMENTOS_TOTAL AS DOUBLE), 0) AS TAXA_FATURA_ABERTA,
    a.QTD_PAGAMENTOS_COM_JUROS / NULLIF(CAST(a.QTD_PAGAMENTOS_TOTAL AS DOUBLE), 0) AS TAXA_PAGAMENTOS_COM_JUROS,
    a.VLR_JUROS_MULTAS_TOTAL / NULLIF(a.VLR_PAGAMENTO_FATURA_TOTAL, 0) AS TAXA_JUROS_SOBRE_PAGAMENTO,
    a.VLR_PAGAMENTO_FATURA_STDDEV / NULLIF(a.VLR_PAGAMENTO_FATURA_MEDIO, 0) AS COEF_VARIACAO_PAGAMENTO,
    a.VLR_PAGAMENTO_FATURA_MAX / NULLIF(a.VLR_PAGAMENTO_FATURA_TOTAL, 0) AS INDICE_CONCENTRACAO,
    a.VLR_PAGAMENTO_FATURA_TOTAL / NULLIF(a.QTD_CONTRATOS, 0) AS VLR_TICKET_MEDIO_CONTRATO,
    a.VLR_ORIGINAL_PAGAMENTO_TOTAL - a.VLR_ATUAL_PAGAMENTO_TOTAL AS VLR_DIFERENCA_ORIG_ATUAL,
    LEAST(100, GREATEST(0,
        COALESCE(a.QTD_FATURA_ABERTA / NULLIF(CAST(a.QTD_PAGAMENTOS_TOTAL AS DOUBLE), 0), 0) * 25 +
        LEAST(1, COALESCE(a.VLR_JUROS_MULTAS_TOTAL / NULLIF(a.VLR_PAGAMENTO_FATURA_TOTAL, 0), 0) * 5) * 25 +
        COALESCE(a.QTD_STATUS_B / NULLIF(CAST(a.QTD_PAGAMENTOS_TOTAL AS DOUBLE), 0), 0) * 20 +
        LEAST(1, COALESCE(a.VLR_PAGAMENTO_FATURA_STDDEV / NULLIF(a.VLR_PAGAMENTO_FATURA_MEDIO, 0), 0)) * 15 +
        CASE WHEN a.QTD_MULTAS_EQUIP > 0 THEN 15 ELSE 0 END
    )) AS SCORE_RISCO,
    CASE
        WHEN a.QTD_FATURA_ABERTA > a.QTD_FATURA_FECHADA THEN 1
        WHEN a.VLR_JUROS_MULTAS_TOTAL > a.VLR_PAGAMENTO_FATURA_TOTAL * 0.1 THEN 1
        WHEN a.QTD_STATUS_B > a.QTD_PAGAMENTOS_TOTAL * 0.3 THEN 1
        ELSE 0
    END AS FLAG_ALTO_RISCO,
    CASE
        WHEN a.QTD_FATURA_ABERTA = 0
            AND a.VLR_JUROS_MULTAS_TOTAL = 0
            AND a.QTD_STATUS_C > a.QTD_PAGAMENTOS_TOTAL * 0.9 THEN 1
        ELSE 0
    END AS FLAG_BAIXO_RISCO,
    CASE
        WHEN a.QTD_FATURA_ABERTA > a.QTD_FATURA_FECHADA
            OR a.VLR_JUROS_MULTAS_TOTAL > a.VLR_PAGAMENTO_FATURA_TOTAL * 0.2 THEN 'CRITICO'
        WHEN a.QTD_FATURA_ABERTA > 0
            OR a.VLR_JUROS_MULTAS_TOTAL > a.VLR_PAGAMENTO_FATURA_TOTAL * 0.1 THEN 'ALTO'
        WHEN a.VLR_JUROS_MULTAS_TOTAL > 0
            OR a.QTD_STATUS_P > a.QTD_PAGAMENTOS_TOTAL * 0.1 THEN 'MEDIO'
        ELSE 'BAIXO'
    END AS SEGMENTO_RISCO
FROM agregado a
"""

In [ ]:
# Executar Book Pagamento para todas as SAFRAs
print("[GOLD] Construindo book_pagamento...")
df_book_pag = build_book_for_safras(spark, SQL_PAGAMENTO, SAFRAS)

book_pag_uri = f"oci://{gold_bucket}@{NAMESPACE}/books/book_pagamento/"
writer = df_book_pag.write.mode("overwrite").partitionBy("SAFRA")
if STORAGE_FORMAT == "delta":
    writer.format("delta").option("overwriteSchema", "true").save(book_pag_uri)
else:
    writer.parquet(book_pag_uri)

print(f"[GOLD BOOK] book_pagamento: escrito, {len(df_book_pag.columns)} colunas")

# Re-ler do Delta
df_book_pag = spark.read.format("delta").load(book_pag_uri)
print(f"[GOLD] Re-leitura book_pagamento: {len(df_book_pag.columns)} colunas")

## Book Faturamento (~108 variaveis, prefixo FAT_)

Variaveis derivadas de transacoes de faturamento, enriquecidas com dimensoes (plataforma, tipo faturamento).

**Decisao anti-leakage**: Este book **NAO** faz JOIN com `dados_cadastrais` para evitar vazamento de informacao da variavel target (FPD). A coluna `VLR_FPD` e excluida na consolidacao final via `LEAKAGE_BLACKLIST`.

**Categorias de variaveis**:
- Contagens: faturas, contratos, safras distintas, por indicador (WO, PDD, ACA, fraude, isencao)
- Valores: bruto, liquido, credito, aberto, ajuste, multas, parcela aparelho
- Por plataforma: AUTOC, POSPG, PREPG, POSBL, por grupo (Pos Pago, Pre Pago, Controle, Telemetria)
- Taxas e ratios: WO, PDD, atraso, credito/bruto, multa/juros
- Temporais: dias desde primeira/ultima fatura, atraso medio/max/min, desde ativacao conta
- Score de risco e segmentacao

In [ ]:
SQL_FATURAMENTO = """
WITH base_enrich AS (
    SELECT
        f.NUM_CPF,
        CAST('{safra}' AS INT) AS SAFRA,
        f.CONTRATO, f.DAT_REFERENCIA, f.DAT_CRIACAO_FAT,
        f.DAT_VENCIMENTO_FAT, f.DAT_STATUS_FAT, f.DAT_ATIVACAO_CONTA_CLI,
        COALESCE(f.VAL_FAT_BRUTO, 0) AS VAL_FAT_BRUTO,
        COALESCE(f.VAL_FAT_LIQUIDO, 0) AS VAL_FAT_LIQUIDO,
        COALESCE(f.VAL_FAT_CREDITO, 0) AS VAL_FAT_CREDITO,
        COALESCE(f.VAL_FAT_ABERTO, 0) AS VAL_FAT_ABERTO,
        COALESCE(f.VAL_FAT_ABERTO_LIQ, 0) AS VAL_FAT_ABERTO_LIQ,
        COALESCE(f.VAL_FAT_AJUSTE, 0) AS VAL_FAT_AJUSTE,
        COALESCE(f.VAL_FAT_BRUTO_BC, 0) AS VAL_FAT_BRUTO_BC,
        COALESCE(f.VAL_FAT_PAGAMENTO_BRUTO, 0) AS VAL_FAT_PAGAMENTO_BRUTO,
        COALESCE(f.VAL_FAT_LIQ_JM_MC, 0) AS VAL_FAT_LIQ_JM_MC,
        COALESCE(f.VAL_MULTA_JUROS, 0) AS VAL_MULTA_JUROS,
        COALESCE(f.VAL_MULTA_CANCELAMENTO, 0) AS VAL_MULTA_CANCELAMENTO,
        COALESCE(f.VAL_PARC_APARELHO_LIQ, 0) AS VAL_PARC_APARELHO_LIQ,
        f.IND_WO, f.IND_PDD, f.IND_PCCR, f.IND_ACA,
        f.IND_PRIMEIRA_FAT, f.IND_FRAUDE, f.IND_ISENCAO_COB_FAT,
        f.COD_PLATAFORMA,
        p.DSC_GRUPO_PLATAFORMA, p.COD_GRUPO_PLATAFORMA_BI,
        t.DSC_TIPO_FATURAMENTO
    FROM fato_faturamento f
    LEFT JOIN dim_plataforma p ON f.COD_PLATAFORMA = p.COD_PLATAFORMA
    LEFT JOIN dim_tipo_faturamento t ON f.DW_TIPO_FATURAMENTO = t.DW_TIPO_FATURAMENTO
    WHERE f.DAT_REFERENCIA < '{data_cutoff}'
),
agregado AS (
    SELECT
        NUM_CPF, SAFRA,
        COUNT(*) AS QTD_FATURAS,
        COUNT(DISTINCT CONTRATO) AS QTD_CONTRATOS_DISTINTOS,
        COUNT(DISTINCT DAT_REFERENCIA) AS QTD_SAFRAS_DISTINTAS,
        SUM(CASE WHEN IND_PRIMEIRA_FAT = 'S' THEN 1 ELSE 0 END) AS QTD_FATURAS_PRIMEIRA,
        SUM(CASE WHEN IND_WO = 'W' THEN 1 ELSE 0 END) AS QTD_FATURAS_WO,
        SUM(CASE WHEN IND_WO = 'R' THEN 1 ELSE 0 END) AS QTD_FATURAS_REGULAR,
        SUM(CASE WHEN IND_PDD = 'S' THEN 1 ELSE 0 END) AS QTD_FATURAS_PDD,
        SUM(CASE WHEN IND_PDD = 'N' THEN 1 ELSE 0 END) AS QTD_FATURAS_SEM_PDD,
        SUM(CASE WHEN IND_PCCR = 'C' THEN 1 ELSE 0 END) AS QTD_FATURAS_PCCR_C,
        SUM(CASE WHEN IND_PCCR = 'W' THEN 1 ELSE 0 END) AS QTD_FATURAS_PCCR_W,
        SUM(CASE WHEN IND_ACA = 'S' THEN 1 ELSE 0 END) AS QTD_FATURAS_ACA,
        SUM(CASE WHEN IND_ACA = 'N' THEN 1 ELSE 0 END) AS QTD_FATURAS_SEM_ACA,
        SUM(CASE WHEN IND_FRAUDE = 'S' THEN 1 ELSE 0 END) AS QTD_FATURAS_FRAUDE,
        SUM(CASE WHEN IND_ISENCAO_COB_FAT IN ('Y','S') THEN 1 ELSE 0 END) AS QTD_FATURAS_ISENTAS,
        SUM(CASE WHEN IND_ISENCAO_COB_FAT = 'N' THEN 1 ELSE 0 END) AS QTD_FATURAS_NAO_ISENTAS,
        SUM(VAL_FAT_BRUTO) AS VLR_FAT_BRUTO_TOTAL,
        AVG(VAL_FAT_BRUTO) AS VLR_FAT_BRUTO_MEDIO,
        MAX(VAL_FAT_BRUTO) AS VLR_FAT_BRUTO_MAX,
        MIN(VAL_FAT_BRUTO) AS VLR_FAT_BRUTO_MIN,
        STDDEV(VAL_FAT_BRUTO) AS VLR_FAT_BRUTO_STDDEV,
        SUM(VAL_FAT_LIQUIDO) AS VLR_FAT_LIQUIDO_TOTAL,
        AVG(VAL_FAT_LIQUIDO) AS VLR_FAT_LIQUIDO_MEDIO,
        MAX(VAL_FAT_LIQUIDO) AS VLR_FAT_LIQUIDO_MAX,
        SUM(VAL_FAT_CREDITO) AS VLR_FAT_CREDITO_TOTAL,
        AVG(VAL_FAT_CREDITO) AS VLR_FAT_CREDITO_MEDIO,
        MAX(VAL_FAT_CREDITO) AS VLR_FAT_CREDITO_MAX,
        SUM(VAL_FAT_ABERTO) AS VLR_FAT_ABERTO_TOTAL,
        AVG(VAL_FAT_ABERTO) AS VLR_FAT_ABERTO_MEDIO,
        MAX(VAL_FAT_ABERTO) AS VLR_FAT_ABERTO_MAX,
        SUM(VAL_FAT_ABERTO_LIQ) AS VLR_FAT_ABERTO_LIQ_TOTAL,
        AVG(VAL_FAT_ABERTO_LIQ) AS VLR_FAT_ABERTO_LIQ_MEDIO,
        SUM(VAL_MULTA_JUROS) AS VLR_MULTA_JUROS_TOTAL,
        AVG(VAL_MULTA_JUROS) AS VLR_MULTA_JUROS_MEDIO,
        MAX(VAL_MULTA_JUROS) AS VLR_MULTA_JUROS_MAX,
        SUM(VAL_MULTA_CANCELAMENTO) AS VLR_MULTA_CANCEL_TOTAL,
        AVG(VAL_MULTA_CANCELAMENTO) AS VLR_MULTA_CANCEL_MEDIO,
        SUM(VAL_FAT_AJUSTE) AS VLR_FAT_AJUSTE_TOTAL,
        SUM(VAL_FAT_PAGAMENTO_BRUTO) AS VLR_FAT_PAGAMENTO_BRUTO_TOTAL,
        SUM(VAL_PARC_APARELHO_LIQ) AS VLR_PARC_APARELHO_TOTAL,
        SUM(VAL_FAT_LIQ_JM_MC) AS VLR_FAT_LIQ_JM_MC_TOTAL,
        SUM(CASE WHEN COD_PLATAFORMA = 'AUTOC' THEN 1 ELSE 0 END) AS QTD_FAT_AUTOC,
        SUM(CASE WHEN COD_PLATAFORMA = 'AUTOC' THEN VAL_FAT_LIQUIDO ELSE 0 END) AS VLR_FAT_AUTOC,
        SUM(CASE WHEN COD_PLATAFORMA = 'POSPG' THEN 1 ELSE 0 END) AS QTD_FAT_POSPG,
        SUM(CASE WHEN COD_PLATAFORMA = 'POSPG' THEN VAL_FAT_LIQUIDO ELSE 0 END) AS VLR_FAT_POSPG,
        SUM(CASE WHEN COD_PLATAFORMA = 'PREPG' THEN 1 ELSE 0 END) AS QTD_FAT_PREPG,
        SUM(CASE WHEN COD_PLATAFORMA = 'PREPG' THEN VAL_FAT_LIQUIDO ELSE 0 END) AS VLR_FAT_PREPG,
        SUM(CASE WHEN COD_PLATAFORMA = 'POSBL' THEN 1 ELSE 0 END) AS QTD_FAT_POSBL,
        SUM(CASE WHEN COD_PLATAFORMA = 'POSBL' THEN VAL_FAT_LIQUIDO ELSE 0 END) AS VLR_FAT_POSBL,
        SUM(CASE WHEN COD_PLATAFORMA = '-2' THEN 1 ELSE 0 END) AS QTD_FAT_SEM_PLATAFORMA,
        SUM(CASE WHEN COD_PLATAFORMA = '-2' THEN VAL_FAT_LIQUIDO ELSE 0 END) AS VLR_FAT_SEM_PLATAFORMA,
        SUM(CASE WHEN DSC_GRUPO_PLATAFORMA = 'Pos Pago' THEN 1 ELSE 0 END) AS QTD_FAT_POS_PAGO,
        SUM(CASE WHEN DSC_GRUPO_PLATAFORMA = 'Pos Pago' THEN VAL_FAT_LIQUIDO ELSE 0 END) AS VLR_FAT_POS_PAGO,
        SUM(CASE WHEN DSC_GRUPO_PLATAFORMA = 'Pre Pago' THEN 1 ELSE 0 END) AS QTD_FAT_PRE_PAGO,
        SUM(CASE WHEN DSC_GRUPO_PLATAFORMA = 'Pre Pago' THEN VAL_FAT_LIQUIDO ELSE 0 END) AS VLR_FAT_PRE_PAGO,
        SUM(CASE WHEN DSC_GRUPO_PLATAFORMA = 'Controle' THEN 1 ELSE 0 END) AS QTD_FAT_CONTROLE,
        SUM(CASE WHEN DSC_GRUPO_PLATAFORMA = 'Controle' THEN VAL_FAT_LIQUIDO ELSE 0 END) AS VLR_FAT_CONTROLE,
        SUM(CASE WHEN DSC_GRUPO_PLATAFORMA = 'Telemetria' THEN 1 ELSE 0 END) AS QTD_FAT_TELEMETRIA,
        SUM(CASE WHEN DSC_GRUPO_PLATAFORMA = 'Telemetria' THEN VAL_FAT_LIQUIDO ELSE 0 END) AS VLR_FAT_TELEMETRIA,
        SUM(CASE WHEN DSC_GRUPO_PLATAFORMA = 'Outros' THEN 1 ELSE 0 END) AS QTD_FAT_OUTROS,
        SUM(CASE WHEN DSC_GRUPO_PLATAFORMA = 'Outros' THEN VAL_FAT_LIQUIDO ELSE 0 END) AS VLR_FAT_OUTROS,
        SUM(CASE WHEN IND_WO = 'W' THEN VAL_FAT_LIQUIDO ELSE 0 END) AS VLR_FAT_WO,
        SUM(CASE WHEN IND_WO = 'R' THEN VAL_FAT_LIQUIDO ELSE 0 END) AS VLR_FAT_REGULAR,
        SUM(CASE WHEN IND_PDD = 'S' THEN VAL_FAT_LIQUIDO ELSE 0 END) AS VLR_FAT_PDD,
        SUM(CASE WHEN IND_PDD = 'N' THEN VAL_FAT_LIQUIDO ELSE 0 END) AS VLR_FAT_SEM_PDD,
        SUM(CASE WHEN IND_ACA = 'S' THEN VAL_FAT_LIQUIDO ELSE 0 END) AS VLR_FAT_ACA,
        SUM(CASE WHEN IND_FRAUDE = 'S' THEN VAL_FAT_LIQUIDO ELSE 0 END) AS VLR_FAT_FRAUDE,
        SUM(CASE WHEN IND_PRIMEIRA_FAT = 'S' THEN VAL_FAT_LIQUIDO ELSE 0 END) AS VLR_PRIMEIRA_FAT,
        AVG(CASE WHEN IND_WO = 'W' THEN VAL_FAT_LIQUIDO END) AS VLR_MEDIO_WO,
        AVG(CASE WHEN IND_PDD = 'S' THEN VAL_FAT_LIQUIDO END) AS VLR_MEDIO_PDD,
        AVG(CASE WHEN IND_PRIMEIRA_FAT = 'S' THEN VAL_FAT_LIQUIDO END) AS VLR_MEDIO_PRIMEIRA_FAT,
        MAX(CASE WHEN IND_WO = 'W' THEN VAL_FAT_LIQUIDO END) AS VLR_MAX_WO,
        MAX(CASE WHEN IND_PDD = 'S' THEN VAL_FAT_LIQUIDO END) AS VLR_MAX_PDD,
        DATEDIFF(TO_DATE('{data_cutoff}'), MIN(DAT_CRIACAO_FAT)) AS DIAS_DESDE_PRIMEIRA_FAT,
        DATEDIFF(TO_DATE('{data_cutoff}'), MAX(DAT_CRIACAO_FAT)) AS DIAS_DESDE_ULTIMA_FAT,
        DATEDIFF(MAX(DAT_CRIACAO_FAT), MIN(DAT_CRIACAO_FAT)) AS DIAS_ENTRE_PRIMEIRA_ULTIMA_FAT,
        DATEDIFF(TO_DATE('{data_cutoff}'), MIN(DAT_ATIVACAO_CONTA_CLI)) AS DIAS_DESDE_ATIVACAO_CONTA,
        AVG(DATEDIFF(DAT_VENCIMENTO_FAT, DAT_CRIACAO_FAT)) AS DIAS_MEDIO_CRIACAO_VENCIMENTO,
        MAX(DATEDIFF(DAT_VENCIMENTO_FAT, DAT_CRIACAO_FAT)) AS DIAS_MAX_CRIACAO_VENCIMENTO,
        MIN(DATEDIFF(DAT_VENCIMENTO_FAT, DAT_CRIACAO_FAT)) AS DIAS_MIN_CRIACAO_VENCIMENTO,
        AVG(DATEDIFF(DAT_STATUS_FAT, DAT_VENCIMENTO_FAT)) AS DIAS_ATRASO_MEDIO,
        MAX(DATEDIFF(DAT_STATUS_FAT, DAT_VENCIMENTO_FAT)) AS DIAS_ATRASO_MAX,
        MIN(DATEDIFF(DAT_STATUS_FAT, DAT_VENCIMENTO_FAT)) AS DIAS_ATRASO_MIN,
        SUM(CASE WHEN DATEDIFF(DAT_STATUS_FAT, DAT_VENCIMENTO_FAT) > 0 THEN 1 ELSE 0 END) AS QTD_FATURAS_ATRASADAS,
        SUM(CASE WHEN DATEDIFF(DAT_STATUS_FAT, DAT_VENCIMENTO_FAT) > 30 THEN 1 ELSE 0 END) AS QTD_FATURAS_ATRASO_30D,
        SUM(CASE WHEN DATEDIFF(DAT_STATUS_FAT, DAT_VENCIMENTO_FAT) > 60 THEN 1 ELSE 0 END) AS QTD_FATURAS_ATRASO_60D,
        SUM(CASE WHEN DATEDIFF(DAT_STATUS_FAT, DAT_VENCIMENTO_FAT) > 90 THEN 1 ELSE 0 END) AS QTD_FATURAS_ATRASO_90D,
        SUM(CASE WHEN DATEDIFF(DAT_STATUS_FAT, DAT_VENCIMENTO_FAT) <= 0 THEN 1 ELSE 0 END) AS QTD_FATURAS_EM_DIA
    FROM base_enrich
    GROUP BY NUM_CPF, SAFRA
)
SELECT
    a.*,
    ROUND(a.QTD_FATURAS_WO / NULLIF(a.QTD_FATURAS, 0), 4) AS TAXA_WO,
    ROUND(a.QTD_FATURAS_PDD / NULLIF(a.QTD_FATURAS, 0), 4) AS TAXA_PDD,
    ROUND(a.QTD_FATURAS_ACA / NULLIF(a.QTD_FATURAS, 0), 4) AS TAXA_ACA,
    ROUND(a.QTD_FATURAS_FRAUDE / NULLIF(a.QTD_FATURAS, 0), 4) AS TAXA_FRAUDE,
    ROUND(a.QTD_FATURAS_PRIMEIRA / NULLIF(a.QTD_FATURAS, 0), 4) AS TAXA_PRIMEIRA_FAT,
    ROUND(a.QTD_FATURAS_ISENTAS / NULLIF(a.QTD_FATURAS, 0), 4) AS TAXA_ISENCAO,
    ROUND(a.QTD_FATURAS_ATRASADAS / NULLIF(a.QTD_FATURAS, 0), 4) AS TAXA_ATRASO,
    ROUND(a.QTD_FATURAS_ATRASO_30D / NULLIF(a.QTD_FATURAS, 0), 4) AS TAXA_ATRASO_30D,
    ROUND(a.QTD_FATURAS_ATRASO_60D / NULLIF(a.QTD_FATURAS, 0), 4) AS TAXA_ATRASO_60D,
    ROUND(a.QTD_FATURAS_ATRASO_90D / NULLIF(a.QTD_FATURAS, 0), 4) AS TAXA_ATRASO_90D,
    ROUND(a.VLR_FAT_CREDITO_TOTAL / NULLIF(a.VLR_FAT_BRUTO_TOTAL, 0), 4) AS TAXA_CREDITO_BRUTO,
    ROUND(a.VLR_FAT_ABERTO_TOTAL / NULLIF(a.VLR_FAT_BRUTO_TOTAL, 0), 4) AS TAXA_ABERTO_BRUTO,
    ROUND(a.VLR_MULTA_JUROS_TOTAL / NULLIF(a.VLR_FAT_LIQUIDO_TOTAL, 0), 4) AS TAXA_MULTA_JUROS,
    ROUND(a.VLR_FAT_WO / NULLIF(a.VLR_FAT_LIQUIDO_TOTAL, 0), 4) AS TAXA_VLR_WO,
    ROUND(a.VLR_FAT_PDD / NULLIF(a.VLR_FAT_LIQUIDO_TOTAL, 0), 4) AS TAXA_VLR_PDD,
    ROUND(a.VLR_FAT_BRUTO_STDDEV / NULLIF(a.VLR_FAT_BRUTO_MEDIO, 0), 4) AS COEF_VAR_FAT_BRUTO,
    ROUND(a.VLR_FAT_BRUTO_MAX / NULLIF(a.VLR_FAT_BRUTO_TOTAL, 0), 4) AS INDICE_CONCENTRACAO_FAT,
    ROUND(GREATEST(a.VLR_FAT_AUTOC, a.VLR_FAT_POSPG, a.VLR_FAT_PREPG, a.VLR_FAT_POSBL) / NULLIF(a.VLR_FAT_LIQUIDO_TOTAL, 0), 4) AS INDICE_CONCENTRACAO_PLATAFORMA,
    ROUND((a.VLR_FAT_BRUTO_MAX - a.VLR_FAT_BRUTO_MIN) / NULLIF(a.VLR_FAT_BRUTO_MEDIO, 0), 4) AS AMPLITUDE_RELATIVA_FAT,
    ROUND(a.VLR_FAT_POS_PAGO / NULLIF(a.VLR_FAT_LIQUIDO_TOTAL, 0), 4) AS SHARE_WALLET_POS_PAGO,
    ROUND(a.VLR_FAT_PRE_PAGO / NULLIF(a.VLR_FAT_LIQUIDO_TOTAL, 0), 4) AS SHARE_WALLET_PRE_PAGO,
    ROUND(a.VLR_FAT_CONTROLE / NULLIF(a.VLR_FAT_LIQUIDO_TOTAL, 0), 4) AS SHARE_WALLET_CONTROLE,
    ROUND(a.VLR_FAT_TELEMETRIA / NULLIF(a.VLR_FAT_LIQUIDO_TOTAL, 0), 4) AS SHARE_WALLET_TELEMETRIA,
    ROUND(LEAST(100, GREATEST(0,
        (COALESCE(a.QTD_FATURAS_WO / NULLIF(a.QTD_FATURAS, 0), 0) * 30) +
        (COALESCE(a.QTD_FATURAS_PDD / NULLIF(a.QTD_FATURAS, 0), 0) * 25) +
        (COALESCE(a.QTD_FATURAS_ATRASO_30D / NULLIF(a.QTD_FATURAS, 0), 0) * 20) +
        (COALESCE(a.QTD_FATURAS_FRAUDE / NULLIF(a.QTD_FATURAS, 0), 0) * 15) +
        (COALESCE(a.QTD_FATURAS_ACA / NULLIF(a.QTD_FATURAS, 0), 0) * 10)
    ) * 100), 2) AS SCORE_RISCO,
    CASE WHEN (COALESCE(a.QTD_FATURAS_WO / NULLIF(a.QTD_FATURAS, 0), 0) * 30 +
               COALESCE(a.QTD_FATURAS_PDD / NULLIF(a.QTD_FATURAS, 0), 0) * 25 +
               COALESCE(a.QTD_FATURAS_ATRASO_30D / NULLIF(a.QTD_FATURAS, 0), 0) * 20 +
               COALESCE(a.QTD_FATURAS_FRAUDE / NULLIF(a.QTD_FATURAS, 0), 0) * 15 +
               COALESCE(a.QTD_FATURAS_ACA / NULLIF(a.QTD_FATURAS, 0), 0) * 10) * 100 >= 70 THEN 1 ELSE 0 END AS FLAG_ALTO_RISCO,
    CASE WHEN (COALESCE(a.QTD_FATURAS_WO / NULLIF(a.QTD_FATURAS, 0), 0) * 30 +
               COALESCE(a.QTD_FATURAS_PDD / NULLIF(a.QTD_FATURAS, 0), 0) * 25 +
               COALESCE(a.QTD_FATURAS_ATRASO_30D / NULLIF(a.QTD_FATURAS, 0), 0) * 20 +
               COALESCE(a.QTD_FATURAS_FRAUDE / NULLIF(a.QTD_FATURAS, 0), 0) * 15 +
               COALESCE(a.QTD_FATURAS_ACA / NULLIF(a.QTD_FATURAS, 0), 0) * 10) * 100 < 30 THEN 1 ELSE 0 END AS FLAG_BAIXO_RISCO,
    CASE
        WHEN (COALESCE(a.QTD_FATURAS_WO / NULLIF(a.QTD_FATURAS, 0), 0) * 30 +
              COALESCE(a.QTD_FATURAS_PDD / NULLIF(a.QTD_FATURAS, 0), 0) * 25 +
              COALESCE(a.QTD_FATURAS_ATRASO_30D / NULLIF(a.QTD_FATURAS, 0), 0) * 20 +
              COALESCE(a.QTD_FATURAS_FRAUDE / NULLIF(a.QTD_FATURAS, 0), 0) * 15 +
              COALESCE(a.QTD_FATURAS_ACA / NULLIF(a.QTD_FATURAS, 0), 0) * 10) * 100 >= 80 THEN 'CRITICO'
        WHEN (COALESCE(a.QTD_FATURAS_WO / NULLIF(a.QTD_FATURAS, 0), 0) * 30 +
              COALESCE(a.QTD_FATURAS_PDD / NULLIF(a.QTD_FATURAS, 0), 0) * 25 +
              COALESCE(a.QTD_FATURAS_ATRASO_30D / NULLIF(a.QTD_FATURAS, 0), 0) * 20 +
              COALESCE(a.QTD_FATURAS_FRAUDE / NULLIF(a.QTD_FATURAS, 0), 0) * 15 +
              COALESCE(a.QTD_FATURAS_ACA / NULLIF(a.QTD_FATURAS, 0), 0) * 10) * 100 >= 50 THEN 'ALTO'
        WHEN (COALESCE(a.QTD_FATURAS_WO / NULLIF(a.QTD_FATURAS, 0), 0) * 30 +
              COALESCE(a.QTD_FATURAS_PDD / NULLIF(a.QTD_FATURAS, 0), 0) * 25 +
              COALESCE(a.QTD_FATURAS_ATRASO_30D / NULLIF(a.QTD_FATURAS, 0), 0) * 20 +
              COALESCE(a.QTD_FATURAS_FRAUDE / NULLIF(a.QTD_FATURAS, 0), 0) * 15 +
              COALESCE(a.QTD_FATURAS_ACA / NULLIF(a.QTD_FATURAS, 0), 0) * 10) * 100 >= 30 THEN 'MEDIO'
        ELSE 'BAIXO'
    END AS SEGMENTO_RISCO
FROM agregado a
"""

In [ ]:
# Executar Book Faturamento para todas as SAFRAs
print("[GOLD] Construindo book_faturamento...")
df_book_fat = build_book_for_safras(spark, SQL_FATURAMENTO, SAFRAS)

book_fat_uri = f"oci://{gold_bucket}@{NAMESPACE}/books/book_faturamento/"
writer = df_book_fat.write.mode("overwrite").partitionBy("SAFRA")
if STORAGE_FORMAT == "delta":
    writer.format("delta").option("overwriteSchema", "true").save(book_fat_uri)
else:
    writer.parquet(book_fat_uri)

print(f"[GOLD BOOK] book_faturamento: escrito, {len(df_book_fat.columns)} colunas")

# Re-ler do Delta
df_book_fat = spark.read.format("delta").load(book_fat_uri)
print(f"[GOLD] Re-leitura book_faturamento: {len(df_book_fat.columns)} colunas")

## Validacao dos Gold Books

Verificacoes dos 3 books escritos no Gold:
- Existencia e row count > 0
- Prefixos corretos (REC_, PAG_, FAT_)
- PK uniqueness em (NUM_CPF, SAFRA)

In [ ]:
# Validacao dos Gold Books
from pyspark.sql.functions import col, count as spark_count

print(f"{\'=\'*60}")
print(f"VALIDACAO — GOLD BOOKS")
print(f"{\'=\'*60}")
checks_pass = 0
checks_fail = 0

books = {
    "book_recarga_cmv":  {"df": df_book_rec,  "prefix": "REC_"},
    "book_pagamento":    {"df": df_book_pag,   "prefix": "PAG_"},
    "book_faturamento":  {"df": df_book_fat,   "prefix": "FAT_"},
}

for book_name, info in books.items():
    df = info["df"]
    prefix = info["prefix"]
    row_count = df.count()
    col_count = len(df.columns)
    prefix_cols = [c for c in df.columns if c.startswith(prefix)]

    # Check: exists and has rows
    if row_count > 0:
        print(f"  [PASS] {book_name}: {row_count:,} rows, {col_count} cols")
        checks_pass += 1
    else:
        print(f"  [FAIL] {book_name}: 0 rows")
        checks_fail += 1

    # Check: prefix columns exist
    if len(prefix_cols) > 0:
        print(f"  [PASS] {book_name}: {len(prefix_cols)} cols com prefixo {prefix}")
        checks_pass += 1
    else:
        print(f"  [FAIL] {book_name}: nenhuma coluna com prefixo {prefix}")
        checks_fail += 1

    # Check: PK uniqueness
    pk = ["NUM_CPF", "SAFRA"]
    dup_count = df.groupBy(*pk).agg(spark_count("*").alias("cnt")).filter(col("cnt") > 1).count()
    if dup_count == 0:
        print(f"  [PASS] {book_name}: PK unique")
        checks_pass += 1
    else:
        print(f"  [FAIL] {book_name}: {dup_count:,} PK duplicates")
        checks_fail += 1

print(f"\n  Resultado: {checks_pass} PASS / {checks_fail} FAIL")
print(f"{\'=\'*60}")